In [1]:
import sys
import csv
import logging  # <-- Ajout du module logging
from selenium import webdriver
import time
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import pandas as pd
import os
from dateutil.relativedelta import relativedelta
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager
import re

# Import pour By
from selenium.webdriver.common.by import By

# Configuration de base du logger
logging.basicConfig(
    level=logging.INFO,  # Pour afficher les logs de niveau INFO et supérieur
    format='%(asctime)s - %(levelname)s - %(message)s'
)

sele = FirefoxService(GeckoDriverManager().install())



2025-03-23 15:13:00,643 - INFO - ====== WebDriver manager ======
2025-03-23 15:13:00,983 - INFO - Get LATEST geckodriver version for 136.0 firefox
2025-03-23 15:13:01,195 - INFO - Get LATEST geckodriver version for 136.0 firefox
2025-03-23 15:13:01,270 - INFO - Driver [/Users/magellan/.wdm/drivers/geckodriver/mac64/v0.36.0/geckodriver] found in cache


In [4]:
def init_driver():
    """
    Initialise le driver Firefox via GeckoDriverManager.
    """
    service = FirefoxService(GeckoDriverManager().install())
    driver = webdriver.Firefox(service=service)
    return driver

In [3]:
def click_privacy_popup(driver):
    """
    Clique sur le pop-up de confidentialité s'il est présent.
    """
    xpath = "/html/body/c-wiz/div/div/div/div[2]/div[1]/div[3]/div[1]/div[1]/form[2]/div/div/button/span[6]"
    try:
        button = driver.find_element(By.XPATH, xpath)
        button.click()
        logging.info("Pop-up de confidentialité cliqué.")
    except Exception as e:
        logging.info(f"Pop-up de confidentialité non trouvé ou déjà fermé : {e}")

def click_sort_button(driver):
    """
    Clique sur le bouton de tri.
    """
    xpath = "/html/body/div[2]/div[3]/div[8]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[4]/div[8]/div[2]/button/span"
    try:
        button = driver.find_element(By.XPATH, xpath)
        button.click()
        logging.info("Bouton de tri cliqué.")
    except Exception as e:
        logging.info(f"Bouton de tri non trouvé : {e}")

def click_recent_reviews(driver):
    """
    Clique sur le bouton pour trier par avis les plus récents.
    """
    xpath = "/html/body/div[2]/div[3]/div[3]/div[1]/div[2]"
    try:
        button = driver.find_element(By.XPATH, xpath)
        button.click()
        logging.info("Bouton 'Avis récents' cliqué.")
    except Exception as e:
        logging.info(f"Bouton 'Avis récents' non trouvé : {e}")

def click_more_buttons(driver):
    """
    Clique sur tous les boutons 'Plus' pour déployer le texte complet des avis.
    """
    try:
        more_buttons = driver.find_elements(By.XPATH, '//button[normalize-space()="Plus"]')
        for idx, btn in enumerate(more_buttons, start=1):
            try:
                btn.click()
                logging.info(f"Bouton 'Plus' n°{idx} cliqué.")
                time.sleep(1)
            except Exception as e:
                logging.info(f"Impossible de cliquer sur le bouton 'Plus' n°{idx} : {e}")
    except Exception as e:
        logging.info(f"Aucun bouton 'Plus' trouvé : {e}")

In [2]:
driver = webdriver.Firefox(service=sele)

In [ ]:
url = "https://www.google.fr/maps/place/Compose+Bois-Colombes+-+Cantine+sur+mesure/@48.9071788,2.2594644,17z/data=!4m8!3m7!1s0x47e665321f7e8b8b:0xe7e1d262385f4e67!8m2!3d48.9071753!4d2.2620393!9m1!1b1!16s%2Fg%2F11tg7p7k1t?entry=ttu&g_ep=EgoyMDI0MTIxMS4wIKXMDSoASAFQAw%3D%3D"
entreprise =  "Compose"
name = "test"
nb_avis = "0"

In [ ]:
driver = webdriver.Firefox(service=sele)

In [ ]:

def get_google_review(url, entreprise, name, nb_avis):
    logging.info(f"Entrée dans get_google_review() pour {entreprise} - {name} - nb_avis={nb_avis}")
    # Import the webdriver
    driver = webdriver.Firefox(service=sele)
    driver.get(url)

    # privacy pop-up
    xpath = "/html/body/c-wiz/div/div/div/div[2]/div[1]/div[3]/div[1]/div[1]/form[2]/div/div/button/span[6]"
    try:
        driver.find_element(By.XPATH, xpath).click()
        logging.info("Pop-up de confidentialité cliquée avec succès.")
    except:
        logging.warning("Pop-up de confidentialité introuvable ou déjà fermée.")

 #   try:
 #       driver.find_element(By.XPATH,
 #           "/html/body/div[3]/div[9]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]/div[1]/div[1]/div[2]/div/div[1]/span[1]/span/span/span[2]/span[1]/button"
 #       ).click()
 #       logging.info("Clic sur le nombre d'avis effectué.")
 #   except:
 #       logging.warning("Impossible de cliquer sur le nombre d'avis.")

    time.sleep(2)

    soup = BeautifulSoup(driver.page_source,"html.parser")

    xpath_nb_avis = "/html/body/div[2]/div[3]/div[8]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[4]/div[1]/div/div[2]/div[3]"
    try:
        total_number_of_reviews_text = driver.find_element(By.XPATH, xpath_nb_avis).text
        total_number_of_reviews = float(total_number_of_reviews_text.split(" ")[-2].replace("\u202f", ""))
        logging.info(f"Nombre total d'avis détectés : {total_number_of_reviews}")
    except Exception as e:
        total_number_of_reviews = 0
        logging.error(f"Erreur lors de la récupération du nombre d'avis : {e}")

    # Catch nombre d'avis
    if nb_avis is not None:
        diff = total_number_of_reviews - float(nb_avis)
        rec_log(entreprise, name, url, total_number_of_reviews, diff)
    else:
        rec_log(entreprise, name, url, float(total_number_of_reviews))

    # Check if there are new comment
    if nb_avis == total_number_of_reviews:
        print("aucun commentaire détecter")
        logging.info("Aucun nouveau commentaire à extraire, on quitte.")
        driver.close()
        return # sys.exit()

    time.sleep(1)
    try:
        xpatrier = "/html/body/div[2]/div[3]/div[8]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[4]/div[8]/div[2]/button/span"
        driver.find_element(By.XPATH, xpatrier).click()
        logging.info("Menu 'Trier' ouvert avec succès.")
    except:
        logging.warning("Échec de l'ouverture du menu 'Trier'.")

    time.sleep(2)
    try:
        xpatrecent = "/html/body/div[2]/div[3]/div[3]/div[1]/div[2]"
        driver.find_element(By.XPATH, xpatrecent).click()
        logging.info("Triage par avis les plus récents cliqué avec succès.")
    except:
        logging.warning("Échec du bouton 'avis les plus récents'.")

    books_html = soup.findAll('div', class_ ="jftiEf fontBodyMedium")
    logging.info(f"Nombre de reviews trouvées avant scrolling : {len(books_html)}")

    #Find scroll layout
    #old_scroll = '//*[@id="pane"]/div/div[1]/div/div/div[2]'
    #old_scroll = "/html/body/div[3]/div[9]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]"
    scroll = "/html/body/div[2]/div[3]/div[8]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]"

    try:
        scrollable_div = driver.find_element(By.XPATH, scroll)
    except:
        scrollable_div = None
        logging.error("Impossible de trouver l'élément scrollable.")

    if nb_avis is not None:
        total_number_of_reviews = total_number_of_reviews - float(nb_avis)

    if scrollable_div and total_number_of_reviews >= 10:
        for i in range(0, (round(total_number_of_reviews / 10 - 1))):
            driver.execute_script('arguments[0].scrollTop = arguments[0].scrollHeight', scrollable_div)
            time.sleep(2)
        logging.info("Scrolling terminé pour charger d'autres avis.")
    else:
        logging.info("Pas de scrolling nécessaire ou scrollable_div introuvable.")

    try:
        liste_plus = driver.find_elements(By.XPATH, '//button[normalize-space()="Plus"]')
        for idx, plus_button in enumerate(liste_plus, start=1):
            try:
                plus_button.click()
            except:
                logging.warning(f"Impossible de cliquer sur le bouton 'Plus' n°{idx}.")
    except:
        logging.warning("Aucun bouton 'Plus' trouvé.")

    response = BeautifulSoup(driver.page_source, 'html.parser')
    reviews = response.find_all("div", class_="jftiEf fontBodyMedium")
    reviews = reviews[:int(total_number_of_reviews)]
    logging.info(f"Nombre de reviews réellement renvoyées : {len(reviews)}")

    driver.close()
    logging.info("Fermeture du navigateur dans get_google_review().")
    return reviews

def get_list_review_google(url, entreprise, name, nb_avis=None):
    logging.info(f"Entrée dans get_list_review_google() pour url={url}, entreprise={entreprise}, name={name}, nb_avis={nb_avis}")
    tmp = get_google_review(url, entreprise, name, nb_avis)
    if tmp is None:
        logging.info("Aucune review récupérée, fin de la fonction.")
        return
    tmp = get_review_summary(tmp)
    tmp = transform_date(tmp)
    tmp["review estimated date"] = [estimated_date(i, j) for i, j in zip(
        tmp["Review Time"], tmp["Review date collected"])]

    tmp = tmp.replace('\|', ',', regex=True)
    filename = entreprise + "_" + name  # Nom pour le fichier CSV

    # Vérifier si le fichier existe
    if os.path.isfile(filename + '.csv'):
        tmp.to_csv(filename + '.csv', sep='|', encoding='utf-8', index=False, mode='a', header=False)
        logging.info(f"{len(tmp)} reviews ajoutées au fichier existant : {filename}.csv")
    else:
        tmp.to_csv(filename + '.csv', sep='|', encoding='utf-8', index=False)
        logging.info(f"Fichier créé : {filename}.csv avec {len(tmp)} reviews.")
    return tmp

def test():
    logging.info("Début de la fonction test().")
    # Chemin vers le fichier CSV
    chemin_fichier = 'log.csv'

    # Charger le fichier CSV avec pandas
    data_frame = pd.read_csv(chemin_fichier)
    logging.info(f"log.csv chargé avec {len(data_frame)} lignes.")

    # Convertir la colonne "date" en type datetime
    data_frame['date'] = pd.to_datetime(data_frame['date'])

    # Trier le dataframe par ordre décroissant de la colonne de date
    data_frame = data_frame.sort_values('date', ascending=False)

    # Regrouper les lignes par les colonnes qui doivent être identiques
    groupes = data_frame.groupby(['entreprise', 'name', 'url'])
    logging.info("Groupby effectué sur (entreprise, name, url).")

    # Sélectionner la ligne la plus récente dans chaque groupe
    lignes_recentes = groupes.apply(
        lambda x: x[x['date'] == x['date'].max()]['nb_avis'])

    # Parcourir les lignes sélectionnées
    for index, nb_avis in lignes_recentes.iteritems():
        entreprise = index[0]
        name = index[1]
        url = index[2]
        logging.info(f"test() - groupe : entreprise={entreprise}, name={name}, url={url}, nb_avis={nb_avis}")
        get_list_review_google(url, entreprise, name, nb_avis)


